# 本地实战P1-01：从参考基线到你自己的RMSNorm/RoPE

概念阅读已由用户确认完成；现在看 [反复练习教程](../docs/LOCAL_PRACTICE.md) 和 [实战日志](../docs/LOCAL_PRACTICE_LOG.md)。
先跑参考CPU/CUDA基线，再编写自己的learning/my_ops.py并指定candidate。
candidate=None只测参考，绝不表示你已独立实现；本Notebook不测性能或模型质量。
下面CUDA单元格明确要求GPU，失败不会静默改用CPU。

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/local_practice.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.local_practice import evaluate, write_report
candidate = None
print('当前实现:', '参考基线' if candidate is None else candidate)

## 1. CPU：先证明计算正确

运行14个固定验收项：Norm边界、非全1gamma、逐token隔离、梯度、RoPE offset/分块/范数及负例。
报告会保存每个case和源码hash。写过自己的文件后，把candidate改为Path('learning/my_ops.py')，重新运行。
加载的是可信本地Python文件；不要把未知网络代码作为candidate。

In [ ]:
cpu_report = evaluate('cpu', candidate)
cpu_path = Path('logs/practice') / ('p1-01-cpu-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
write_report(cpu_report, cpu_path)
print('role:', cpu_report['implementation_role'])
print('passed:', cpu_report['passed'], 'failed:', cpu_report['failed'])
for case in cpu_report['cases']:
    print(case['name'], case['status'], case.get('error', ''))
print('Report:', cpu_path)
assert cpu_report['status'] == 'passed', '先定位CPU失败项，不急着调GPU。'

## 2. CUDA：再证明同一实现能在本机GPU执行

不是只检查cuda.is_available，而是把输入和模块放到CUDA，运行全部同样的验收项。
这里没有吞吐或加速比；P1-01先正确，后续关卡再测性能。
保留环境warning，先按实际执行与对齐结果判断，不升级torch来消除提示。

In [ ]:
gpu_report = evaluate('cuda', candidate)
gpu_path = Path('logs/practice') / ('p1-01-cuda-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
write_report(gpu_report, gpu_path)
print('role:', gpu_report['implementation_role'], 'GPU:', gpu_report['gpu'])
print('passed:', gpu_report['passed'], 'failed:', gpu_report['failed'])
for case in gpu_report['cases']:
    if case['status'] != 'passed':
        print(case['name'], case['error'])
print('Report:', gpu_path)
assert gpu_report['status'] == 'passed'
if candidate is None:
    print('参考基线通过；下一步自己写my_ops.py，不自动登记个人实战完成。')

## 3. 从此反复练习，不只Run All

1. 先遮住参考公式，把自己对shape/offset/gamma的预测写下来。
2. 在learning/my_ops.py实现RMSNorm(width,eps)和rope(x,start,theta)，参数名/布局按教程契约。
3. 用新进程命令传入--candidate实际验收，修复第一个失败case。
4. 故意把offset重置0或把Norm分母轴改错，确认失败能被检测，再修回。
5. 自己增加一个新shape/幅值/低精度边界，不把原14项当作所有输入的证明。
6. 将源码hash、报告和错误原因填入实战日志，再开始单头Attention。

后续21个关卡与验收路径见 [LOCAL_PRACTICE](../docs/LOCAL_PRACTICE.md)，不自动覆盖参考模型或现有笔记。